In [82]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

In [83]:
TEST_SIZE = 0.2

VAL_SIZE = TEST_SIZE / (1 - TEST_SIZE)

RANDOM_STATE = 42

N_SPLITS = 1

In [84]:
df = pd.read_pickle("../data/processed/diabetic_data_encoded.pkl")

### Разделение данных

В датасете 71518 пациентов на 101763 строки - один пациент встречается несколько раз. Обычный train_test_split поместит пациента и в train, и в test, что создает риск утечки данных. Нужен GroupShuffleSplit.

In [85]:
group_col = "patient_nbr"

groups = df[group_col].values

groups

array([  8222157,  55629189,  86047875, ...,  41088789,  31693671,
       175429310], shape=(101763,))

In [86]:
gss_test = GroupShuffleSplit(n_splits=N_SPLITS, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_test.split(df, groups=groups))

train_val = df.iloc[train_val_idx].reset_index(drop=True)
test = df.iloc[test_idx].reset_index(drop=True)

In [87]:
gss_val = GroupShuffleSplit(n_splits=N_SPLITS, test_size=VAL_SIZE, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_val.split(train_val, groups=train_val["patient_nbr"].values))

train = train_val.iloc[train_idx].reset_index(drop=True)
val = train_val.iloc[val_idx].reset_index(drop=True)

In [88]:
print(f"train: {train.shape} ({len(train)/len(df):.2%})")
print(f"val: {val.shape} ({len(val)/len(df):.2%})")
print(f"test: {test.shape} ({len(test)/len(df):.2%})")

print("Доля положительного класса:")
print(f"train: {train['target'].mean():.4f}")
print(f"val: {val['target'].mean():.4f}")
print(f"test: {test['target'].mean():.4f}")

train: (61244, 162) (60.18%)
val: (20426, 162) (20.07%)
test: (20093, 162) (19.74%)
Доля положительного класса:
train: 0.1126
val: 0.1108
test: 0.1092


Удаление id столбцов

In [89]:
id_cols = ["encounter_id"]

train = train.drop(columns=id_cols)
val = val.drop(columns=id_cols)
test = test.drop(columns=id_cols)

Разделим на X и y

In [90]:
def x_y_selection(df, target_col):
    return df.drop(columns=[group_col, target_col]), df[target_col]

In [91]:
target_col = "target"

X_train, y_train = x_y_selection(train, target_col)
X_val, y_val = x_y_selection(val, target_col)
X_test, y_test = x_y_selection(test, target_col)

### Исследование выбросов

### Масштабирование данных